In [1]:
!pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 47.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 15.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 508.3/508.3 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.

In [2]:
from langchain_core.documents import Document

Document -> Object

{

    1. text/page content
    
    2. metadata
    
}

In [3]:
# Text data

from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("/kaggle/input/datasets/radhikaasmar/data-rag/Python.txt", encoding="utf-8")

document = loader.load()
document

[Document(metadata={'source': '/kaggle/input/datasets/radhikaasmar/data-rag/Python.txt'}, page_content='\ufeffPython is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for experienced developers. Its clean and concise syntax allows programmers to write fewer lines of code compared to many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, which makes it versatile for a wide range of applications.\nSome key features and benefits of Python include:\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine lear

In [4]:
# # PDF data

# from langchain_community.document_loaders.pdf import PyPDFLoader

# pdf_loader = PyPDFLoader("/kaggle/input/datasets/radhikaasmar/data-rag/research.pdf")

# document = pdf_loader.load()
# document

In [5]:
# # PDF data

# from langchain_community.document_loaders.pdf import PyMuPDFLoader

# pdf_loader = PyMuPDFLoader("/kaggle/input/datasets/radhikaasmar/data-rag/research.pdf")

# document = pdf_loader.load()
# document

# 1. Injestion Pipeline

In [6]:
# Data => Documents

import os
from langchain_community.document_loaders.pdf import PyPDFLoader

## i. Documents

In [7]:
def load_all_pdfs():
    folder_path = "/kaggle/input/datasets/radhikaasmar/data-rag/pdfs/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [8]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [9]:
type(all_pdf_documents[0])

langchain_core.documents.base.Document

In [10]:
all_pdf_documents[0]

Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On E

## ii. Chunks

In [11]:
# Chunks 

!pip install langchain_text_splitters -q

In [12]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [13]:
chunks = split_docs(all_pdf_documents)
len(chunks)

320

In [14]:
chunks[0]

Document(metadata={'producer': 'PyPDF2', 'creator': 'PyPDF', 'creationdate': '', 'subject': 'Neural Information Processing Systems http://nips.cc/', 'publisher': 'Curran Associates, Inc.', 'language': 'en-US', 'created': '2017', 'eventtype': 'Poster', 'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and requiring significantly less timeto train. Our single model with 165 million parameters, achieves 27.5 BLEU onEnglish-to-German translation, improving over the existing best ensemble result by over 1 BLEU. On E

## iii. Embedding

In [15]:
from sentence_transformers import SentenceTransformer

In [16]:
from sentence_transformers import SentenceTransformer

class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        # Generates numerical representations (embeddings) for the input text
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [17]:
# Initialize the manager
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

embedding dimensions= 384


## iv. Vector Store 

Small local version of Vector DB

* initialize vector store - constructor
* vs_initialize -> path, collection
* store docs -> Vector Store

In [18]:
!pip install -U --force-reinstall chromadb opentelemetry-api opentelemetry-sdk opentelemetry-exporter-otlp -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 3.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.7/57.7 kB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.8/6.8 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.5/73.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.6/90.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 78.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 463.6/463.6 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2

In [19]:
import chromadb
import uuid
import opentelemetry.context
import uuid

In [20]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        # Ensure the directory where data will be saved actually exists
        os.makedirs(self.persist_directory, exist_ok=True)

        # Create a client that saves data to the local hard drive
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # Create the collection (or open it if it already exists)
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())


    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")
        
        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for every single chunk
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)
            
            # Prepare metadata (useful for filtering later)
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)
            
            # Extract the text and convert embedding to list format
            documents_content.append(doc.page_content)
            embeddings_list.append(embedding.tolist())
            
        # Final step: Upload everything to the ChromaDB collection
        self.collection.add(
            ids=ids,
            metadatas=all_metadata,
            documents=documents_content,
            embeddings=embeddings_list
        )
        
        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())


In [21]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 0


In [22]:
# data => documents => chunks => embeddings => store in vector store

# 1. Extract the raw text from your document chunks
texts = [doc.page_content for doc in chunks]

# 2. Use your EmbeddingManager to turn those texts into numerical vectors
embedding = embedding_manager.generate_embeddings(texts)

# 3. Save both the original chunks and their new embeddings into the database
vector_store.add_documents(chunks, embedding)

Batches:   0%|          | 0/10 [00:00<?, ?it/s]

embeddings shape: (320, 384)
total documents added in vector store= 320
docs in collection: 320


# 2. Retrieval Pipeline

In [23]:
from sklearn.metrics.pairwise import cosine_similarity

In [24]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store):
        # Store the managers we built earlier so we can use their methods
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store

    def retrieve(self, query, top_k=5, score_threshold=0.0):
        """
        Searches the vector database for the most relevant document chunks.
        
        Args:
            query (str): The user's natural language question.
            top_k (int): Number of documents to return.
            score_threshold (float): Minimum similarity (0 to 1) to be considered a match.
        """
        
        # 1. Convert the search query into a vector (embedding)
        # We pass [query] as a list and take the first result [0]
        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        # 2. Perform the semantic search in ChromaDB
        # We convert the numpy array to a list so ChromaDB can read it
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=top_k
        )

        retrieved_docs = []
        
        # 3. Check if we actually found anything
        if results["documents"] and results["documents"][0]:
            # ChromaDB returns nested lists; we grab the inner lists
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]

            # 4. Loop through results and calculate a similarity score
            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                
                # Distance measures how 'far apart' vectors are. 
                # Similarity (1 - distance) measures how 'close' they are.
                similarity_score = 1 - distance

                # Only keep the document if it meets our quality bar (threshold)
                if similarity_score >= score_threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1  # 1 is the best match, 2 is the second best, etc.
                    })

            print(f"retrieved {len(retrieved_docs)} documents")

        else:
            print("no documents found")

        return retrieved_docs

In [25]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [26]:
rag_retriever.retrieve("What is RAG?")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_6baf5e6d-21ad-4db4-913c-dc295228892e',
  'document': 'and speculate on upcoming trends and innovations.\nOur contributions are as follows:\n• In this survey, we present a thorough and systematic\nreview of the state-of-the-art RAG methods, delineating\nits evolution through paradigms including naive RAG,\narXiv:2312.10997v5  [cs.CL]  27 Mar 2024',
  'metadata': {'doc_index': 87,
   'author': '',
   'producer': 'pdfTeX-1.40.25',
   'title': '',
   'keywords': '',
   'content_length': 288,
   'creationdate': '2024-03-28T00:54:45+00:00',
   'moddate': '2024-03-28T00:54:45+00:00',
   'page': 0,
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'trapped': '/False',
   'creator': 'LaTeX with hyperref',
   'page_label': '1',
   'source': '/kaggle/input/datasets/radhikaasmar/data-rag/pdfs/pdfs/research2.pdf',
   'total_pages': 21,
   'subject': ''},
  'distance': 0.46291497349739075,
  'similarity_score': 0.537085026

In [27]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 4 documents


[{'id': 'doc_631032d4-a3cb-4ddd-854d-007b89f826ff',
  'document': 'positional encodings in both the encoder and decoder stacks. For the base model, we use a rate of\nPdrop = 0.1.\n7',
  'metadata': {'content_length': 112,
   'description': 'Paper accepted and presented at the Neural Information Processing Systems Conference (http://nips.cc/)',
   'firstpage': '5998',
   'creator': 'PyPDF',
   'eventtype': 'Poster',
   'doc_index': 49,
   'date': '2017',
   'description-abstract': 'The dominant sequence transduction models are based on complex recurrent orconvolutional neural networks in an encoder and decoder configuration. The best performing such models also connect the encoder and decoder through an attentionm echanisms.  We propose a novel, simple network architecture based solely onan attention mechanism, dispensing with recurrence and convolutions entirely.Experiments on two machine translation tasks show these models to be superiorin quality while being more parallelizable and r

# 3. Integrate with LLMs

## 3.1 OpenAI-GPT

In [28]:
!pip install langchain-openai -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 34.9 MB/s eta 0:00:00


In [29]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
API_KEY_OPENAI = user_secrets.get_secret("OPENAI_KEY")

In [30]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    openai_api_key=API_KEY_OPENAI,
    model="gpt-5.4",  
    
    temperature=0.1,    
    # for factual: 0.0 - 0.3
    # balance: 0.4 - 0.7
    # creative: 0.8 - 1.0+

    max_tokens=1024
)

In [31]:
# Generate our retrieval-augmented output
def generate_output(query, rag_retriever, llm, top_k=3):
    # 1. Fetch relevant documents from the vector store
    results = rag_retriever.retrieve(query, top_k)
    
    # 2. Extract content and join into a single string
    context = "\n".join([doc['document'] for doc in results]) if results else ""
    
    # 3. Guardrail for empty context
    if not context:
        print("we found no relevant context for the given query")
        
    # 4. Construct the prompt with context and query
    prompt = f"""
    Use the given context to generate the answer for the query.
    Context: {context}
    Query: {query}
    """
    
    # 5. Get the response from the LLM
    response = llm.invoke(prompt) # expecting a STRING as prompt
    
    return response.content

In [32]:
answer1 = generate_output("What is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [33]:
print(answer1)

RAG stands for Retrieval-Augmented Generation.

It is a framework that improves large language model generation by first retrieving relevant external information, then using that information to generate a response. In simple terms, instead of relying only on what the model memorized during training, RAG lets the model look up useful knowledge and incorporate it into its answer.

From the context, RAG is presented as a major research area in LLMs, with multiple evolving paradigms such as:
- naive RAG
- advanced RAG
- modular RAG

Its goal is to make generated answers more accurate, up-to-date, and grounded in retrieved evidence.


In [34]:
answer2 = generate_output("What is encoder-decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [35]:
print(answer2)

An encoder-decoder is an architecture with two parts:

- Encoder: reads the input and converts it into a sequence of internal representations.
- Decoder: uses those encoder representations to generate the output sequence.

From the context:
- The encoder has N = 6 identical layers.
- Each encoder layer contains:
  - a multi-head self-attention sub-layer
  - a position-wise feed-forward sub-layer
- The decoder also has N = 6 identical layers.
- Each decoder layer contains:
  - a masked self-attention sub-layer
  - an encoder-decoder attention sub-layer that attends to the encoder’s output
  - a position-wise feed-forward sub-layer

Both encoder and decoder use residual connections and layer normalization.

So, encoder-decoder means a model where one network encodes the input, and another decodes that representation into the desired output.


## 3.2 Groq 

In [36]:
!pip install langchain-groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.4 MB/s eta 0:00:00


In [37]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
API_KEY_GROQ = user_secrets.get_secret("GROQ_API")

In [38]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    groq_api_key=API_KEY_GROQ,
    model="qwen/qwen3-32b",  
    temperature=0.1,    
    max_tokens=1024
)

In [39]:
# Generate our retrieval-augmented output
def generate_output(query, rag_retriever, llm, top_k=3):
    results = rag_retriever.retrieve(query, top_k)
    
    context = "\n".join([doc['document'] for doc in results]) if results else ""
    
    if not context:
        print("we found no relevant context for the given query")
        
    prompt = f"""
    Use the given context to generate the answer for the query.
    Context: {context}
    Query: {query}
    """
    
    response = llm.invoke([prompt.format(context=context, query=query)]) # expecting a LIST as prompt
    
    return response.content

In [40]:
answer3 = generate_output("What is RAG?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [41]:
print(answer3)

<think>
Okay, the user is asking, "What is RAG?" I need to use the provided context to answer this. Let me read through the context again to make sure I understand what's relevant.

The context mentions that the paper is a survey on RAG methods, covering their evolution through paradigms like naive RAG and effective RAG frameworks. It also talks about evaluation methods, tasks, datasets, and future directions. The paper is structured with sections on main concepts, paradigms, and integration with LLMs.

So, RAG stands for Retrieval-Augmented Generation. From the context, it's a method that combines retrieval of information with generation, likely using large language models (LLMs). The survey discusses different paradigms of RAG, such as naive RAG and more effective frameworks. The key points to include are that RAG enhances LLMs by retrieving external information to improve responses, and it's an area with rapid growth but needs systematic synthesis. The answer should define RAG, ment

In [42]:
answer4 = generate_output("What is encoder-decoder?", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 3 documents


In [43]:
print(answer4)

<think>
Okay, the user is asking "What is encoder-decoder?" Let me start by recalling the context provided. The context talks about the encoder and decoder stacks in a model, probably a transformer since it mentions multi-head attention and layers.

First, I need to define what an encoder-decoder architecture is. From the context, the encoder has N=6 layers, each with two sub-layers: multi-head self-attention and a position-wise feed-forward network. The decoder also has N=6 layers but includes a third sub-layer for multi-head attention over the encoder's output. Both use residual connections and layer normalization. The model uses d_model=512 and Pdrop=0.1 for dropout.

So, the encoder processes the input, converting it into a context vector. The decoder uses this context to generate the output, attending to the encoder's outputs. The key points are the stacks of layers, the attention mechanisms, and the specific components in each part. I should explain the encoder-decoder structure 